In [1]:
import os
import torch
from transformers import AutoTokenizer,AutoModel,AutoModelForMaskedLM,AutoModelForSequenceClassification

torch.manual_seed(42)
bert_path=r"D:\11\NLP\data\bertbert-base-chinese"
bert_ready=os.path.isdir(bert_path) and os.path.isfile(os.path.join(bert_path,"config.json")) and any(os.path.isfile(os.path.join(bert_path,name)) for name in ["model.safetensors","pytorch_model.bin"])
print("BERT目录：",bert_path); print("模型准备完成：",bert_ready)
if os.path.isdir(bert_path): print("目录文件：",os.listdir(bert_path))

C:\Users\Administrator\.conda\envs\rl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BERT目录： D:\11\NLP\data\bertbert-base-chinese
模型准备完成： True
目录文件： ['config.json', 'model.safetensors', 'tokenizer_config.json', 'vocab.txt']


## 1. 加载本地 BERT Encoder

In [2]:
if bert_ready:
    tokenizer=AutoTokenizer.from_pretrained(bert_path,local_files_only=True)
    model=AutoModel.from_pretrained(bert_path,local_files_only=True,output_hidden_states=True,output_attentions=True)
    model.eval(); print(model.config)
else: print("请先下载本地BERT模型。")
'''参数 output_hidden_states=True 和 output_attentions=True：相当于给这个大脑装上了 “透视镜”，让它把每一层的思考过程（中间状态）和注意力焦点都吐出来给你看。'''

BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "directionality": "bidi",
  "dtype": "float32",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_attentions": true,
  "output_hidden_states": true,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers": 3,
  "pooler_size_per_head": 128,
  "pooler_type": "first_token_transform",
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.6",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 21128
}



## 2. Tokenization 中间结果

In [3]:
text="自然语言处理很有趣"
if bert_ready:
    inputs=tokenizer(text,return_tensors="pt")
    '''分词器处理：tokenizer(text, return_tensors="pt") 做了三件事：
加特殊标记：自动在开头加上 [CLS]（分类标记），结尾加上 [SEP]（分隔标记）。
查字典：把每个字/词映射成数字 ID（比如 "自" → 5632，"然" → 4197）。
转为 PyTorch 张量：return_tensors="pt" 让输出变成 PyTorch 能计算的张量格式。'''
    tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    '''转换成人类可读的 Token：convert_ids_to_tokens 把数字 ID 反向映射回文字，方便肉眼核对分词对不对。'''
    print("原始文本：",text); print("Tokens：",tokens); print("input_ids：",inputs["input_ids"]); print("attention_mask：",inputs["attention_mask"])
else: print("模型未准备。")
'''掩码它是一个“注意力开关”，告诉 BERT 在计算时该看哪里，不该看哪里。
数值含义：1 表示“这是真实的 Token，请重点关注我”；0 表示“这是为了凑长度而补的空白填充符（[PAD]），请彻底无视我，不要浪费注意力”。'''

原始文本： 自然语言处理很有趣
Tokens： ['[CLS]', '自', '然', '语', '言', '处', '理', '很', '有', '趣', '[SEP]']
input_ids： tensor([[ 101, 5632, 4197, 6427, 6241, 1905, 4415, 2523, 3300, 6637,  102]])
attention_mask： tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


## 3. 获取最后隐藏状态、池化输出、全部隐藏层和注意力

In [4]:
if bert_ready:
    with torch.no_grad(): outputs=model(**inputs,output_hidden_states=True,output_attentions=True)
    '''torch.no_grad()：关闭梯度计算。告诉 PyTorch：“我只做推理（前向传播），不做训练（反向传播）”。这会大幅节省显存和计算时间。output_hidden_states=True 和 output_attentions=True：这两个参数是关键中的关键。默认情况下，BERT 只吐出最后一层的结果。加上它们后，BERT 会像切洋葱一样，把每一层的内部状态和注意力权重都吐出来给你。'''
    print("last_hidden_state形状：",outputs.last_hidden_state.shape)
    print("pooler_output形状：",None if outputs.pooler_output is None else outputs.pooler_output.shape)
    print("隐藏状态层数量：",len(outputs.hidden_states)); print("注意力层数量：",len(outputs.attentions))
    print("第一层注意力形状：",outputs.attentions[0].shape)
else: print("模型未准备。")

last_hidden_state形状： torch.Size([1, 11, 768])
pooler_output形状： torch.Size([1, 768])
隐藏状态层数量： 13
注意力层数量： 12
第一层注意力形状： torch.Size([1, 12, 11, 11])


## 4. 查看每个 Token 的 768 维表示

In [5]:
if bert_ready:
    for token,vector in zip(tokens,outputs.last_hidden_state[0]): print(token,"前8维：",vector[:8].tolist())
else: print("模型未准备。")

[CLS] 前8维： [-0.8448424339294434, -0.23739224672317505, -0.10835372656583786, -0.08136110007762909, 0.6706001162528992, -0.9116446375846863, -0.33910107612609863, -0.7820717692375183]
自 前8维： [0.23959441483020782, 0.16769033670425415, 0.873916506767273, 0.43253666162490845, 1.034472942352295, -0.8578442931175232, -0.0761764645576477, -0.2611485719680786]
然 前8维： [-0.007207542657852173, -0.5302804112434387, -0.42444828152656555, -0.4276835322380066, 0.9215224385261536, 0.38640642166137695, 0.41194868087768555, -0.2399216741323471]
语 前8维： [-0.6397922039031982, -0.2365296185016632, 0.8016766905784607, 0.2312701940536499, 0.9934310913085938, 0.14619691669940948, 0.5927801728248596, 0.3005688786506653]
言 前8维： [0.25347989797592163, -0.395345538854599, 0.2614256739616394, -0.7616351246833801, 1.0298525094985962, 0.5021953582763672, 0.8365626335144043, 0.7064204216003418]
处 前8维： [-0.03417566046118736, -0.007973246276378632, 1.357337236404419, 1.1070995330810547, 0.9626622796058655, -0.41698712110

## 5. 使用 [CLS] 作为句向量，并比较句子相似度

In [6]:
sentences=["自然语言处理很有趣","我喜欢学习NLP","今天天气非常晴朗"]
if bert_ready:
    batch=tokenizer(sentences,padding=True,truncation=True,return_tensors="pt")
    with torch.no_grad(): batch_outputs=model(**batch)
    cls_vectors=batch_outputs.last_hidden_state[:,0,:]; normalized=torch.nn.functional.normalize(cls_vectors,p=2,dim=-1); similarity=normalized@normalized.T
    print("句向量形状：",cls_vectors.shape); print("余弦相似度矩阵：\n",similarity)
else: print("模型未准备。")

句向量形状： torch.Size([3, 768])
余弦相似度矩阵：
 tensor([[1.0000, 0.8145, 0.7042],
        [0.8145, 1.0000, 0.6905],
        [0.7042, 0.6905, 1.0000]])


## 6. 查看最后一层第一个注意力头

In [7]:
if bert_ready:
    attention=outputs.attentions[-1][0,0]
    print("注意力矩阵形状：",attention.shape); print("每行权重和：",attention.sum(dim=-1))
    query_index=1
    for token,weight in zip(tokens,attention[query_index]): print(f"{tokens[query_index]} -> {token}: {weight.item():.4f}")
else: print("模型未准备。")

注意力矩阵形状： torch.Size([11, 11])
每行权重和： tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000])
自 -> [CLS]: 0.0040
自 -> 自: 0.0011
自 -> 然: 0.0042
自 -> 语: 0.0011
自 -> 言: 0.0007
自 -> 处: 0.0009
自 -> 理: 0.0009
自 -> 很: 0.0021
自 -> 有: 0.0008
自 -> 趣: 0.0020
自 -> [SEP]: 0.9821


## 7. 句子对和 Segment Embedding

In [8]:
#代码展示了 BERT 最核心的“看家本领”之一：处理句子对（Sentence Pair）。它展示了 BERT 是如何将两句话同时塞进模型，并通过 token_type_ids（段落标记/分段嵌入） 来区分“哪部分是第一句，哪部分是第二句”的。
if bert_ready:
    pair_inputs=tokenizer("自然语言处理很重要","它是人工智能的一个分支",return_tensors="pt")
    print("Tokens：",tokenizer.convert_ids_to_tokens(pair_inputs["input_ids"][0]))
    print("token_type_ids：",pair_inputs.get("token_type_ids"))
    with torch.no_grad(): pair_outputs=model(**pair_inputs)
    print("句子对输出形状：",pair_outputs.last_hidden_state.shape)
else: print("模型未准备。")
'''token_type_ids
0（左半部分）：告诉模型“这些词属于第一句话”。
1（右半部分）：告诉模型“这些词属于第二句话”。'''

'''BERT 处理两句话时，不是把它们分开计算，而是用 [SEP] 拼成一根长串，再用 token_type_ids 给每个词贴上“属于句A”或“属于句B”的标签。 这就像把两封信放进同一个信封，但在信封里的每个字上面盖了一个“第一封信”或“第二封信”的章，方便模型区分归属。'''

Tokens： ['[CLS]', '自', '然', '语', '言', '处', '理', '很', '重', '要', '[SEP]', '它', '是', '人', '工', '智', '能', '的', '一', '个', '分', '支', '[SEP]']
token_type_ids： tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
句子对输出形状： torch.Size([1, 23, 768])


## 8. Masked Language Model：预测 [MASK]

In [9]:
'''给 BERT 出了一道语文填空题：“自然语言 ____ 很有趣”，BERT 不用微调，直接凭借它预训练时读过的海量中文知识，给你填上了 “也”，并给出了每个候选字的置信度。'''
masked_text="自然语言[MASK]很有趣"
if bert_ready:
    mlm_model=AutoModelForMaskedLM.from_pretrained(bert_path,local_files_only=True); mlm_model.eval()
    mlm_inputs=tokenizer(masked_text,return_tensors="pt"); mask_index=(mlm_inputs["input_ids"][0]==tokenizer.mask_token_id).nonzero(as_tuple=True)[0].item()
    with torch.no_grad(): mlm_logits=mlm_model(**mlm_inputs).logits[0,mask_index]
    top_ids=torch.topk(mlm_logits,k=10).indices
    print("输入：",masked_text)
    for token_id in top_ids: print(tokenizer.decode([token_id.item()]),float(torch.softmax(mlm_logits,dim=-1)[token_id]))
else: print("模型未准备。")

Some weights of the model checkpoint at D:\11\NLP\data\bertbert-base-chinese were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


输入： 自然语言[MASK]很有趣
也 0.7000358700752258
， 0.10641234368085861
： 0.04139101132750511
都 0.02509867213666439
会 0.009836910292506218
却 0.008653972297906876
还 0.006387760862708092
就 0.005886303260922432
中 0.004690808709710836
的 0.004659674130380154


## 9. 在本地 BERT 上添加分类层

In [10]:
if bert_ready:
    classification_model=AutoModelForSequenceClassification.from_pretrained(bert_path,num_labels=3,local_files_only=True)
    classification_inputs=tokenizer(["这本书很好看","这个软件很差"],padding=True,return_tensors="pt")
    with torch.no_grad(): classification_logits=classification_model(**classification_inputs).logits
    print("分类Logits形状：",classification_logits.shape); print(classification_logits)
else: print("模型未准备。")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at D:\11\NLP\data\bertbert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


分类Logits形状： torch.Size([2, 3])
tensor([[ 0.2163,  0.1866, -0.6711],
        [ 0.2669,  0.0033, -0.5187]])


预训练模型（如 BERT）是一个已经读过海量中文的“超级大脑”；你可以直接拿它提取特征（透视内部），也可以给它加新脑袋（分类层）并用你的数据教它做新任务（微调）